# Analyze Account Data Sources

We have three sources for account and account holder data:
1. Direct download of the account data
2. Download through the power BI app
3. Inferred from transaction data

We need to examine what are the differences in different source and which information
we take from which of the sources.

## Packages and options

In [1]:
# add the parent directory to the sys.path
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import pandas as pd

In [2]:
fn_direct = "../data/source/automatic/eutl_accounts.csv"
fn_bi = "../data/source/manual/accounts.xlsx"
fn_trans = "../data/source/automatic/eutl_transactions.csv"

## Get data

Account data from direct download:

In [3]:
df_acc_direct = pd.read_csv(fn_direct).assign(
    account_id=lambda df: df["REGISTRY_CODE"]
    + "_"
    + df["ACCOUNT_IDENTIFIER"].astype(str),
)
map_registry_names = df_acc_direct.set_index("REGISTRY_NAME")["REGISTRY_CODE"].to_dict()
map_registry_names.update({"Switzerland": "CH", "CDM": "CDM"})
df_acc_direct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47666 entries, 0 to 47665
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ACCOUNT_IDENTIFIER    47666 non-null  int64 
 1   REGISTRY_CODE         47666 non-null  object
 2   REGISTRY_NAME         47666 non-null  object
 3   ACCOUNT_NAME          47666 non-null  object
 4   ACCOUNT_TYPE          47666 non-null  object
 5   ETS_ACCOUNT_TYPE      28348 non-null  object
 6   FULL_TYPE             47637 non-null  object
 7   OPEN_DATE             47630 non-null  object
 8   END_OF_VALIDITY_DATE  47666 non-null  object
 9   IS_CLOSURE_PENDING    47666 non-null  object
 10  SNAPSHOT_DATE         47666 non-null  object
 11  account_id            47666 non-null  object
dtypes: int64(1), object(11)
memory usage: 4.4+ MB


In [4]:
# df_acc_direct.ACCOUNT_TYPE.unique()

In [5]:
# df_acc_direct.ETS_ACCOUNT_TYPE.unique()

Power BI data

In [6]:
df_acc_bi = (
    pd.read_excel(fn_bi, skipfooter=2)
    .rename(columns={"..1": "registry_id"})
    .drop(columns=".")
    .assign(
        account_id=lambda df: df["registry_id"]
        + "_"
        + df["Account Identifier"].astype(str),
    )
)
df_acc_bi.info()

e:\GIT\eutl_scraper_v2\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47389 entries, 0 to 47388
Data columns (total 15 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Account Identifier                                   47389 non-null  int64  
 1   National Administrator                               47389 non-null  object 
 2   Account Type                                         47389 non-null  object 
 3   Account Holder Name                                  47389 non-null  object 
 4   Account Name                                         47389 non-null  object 
 5   Installation/Aircraft Operator/Maritime Operator ID  22158 non-null  float64
 6   Company Registration No                              42781 non-null  object 
 7   Main Address Line                                    47380 non-null  object 
 8   City                                                 47380 non-nul

Transaction data

In [7]:
df_acc_trans_in = pd.read_csv("../data/source/automatic/eutl_transactions.csv")

C:\Users\abrel\AppData\Local\Temp\ipykernel_34316\313709080.py:1: DtypeWarning: Columns (20,22,23,24,25,26,27,28,29,47,49,50,51,52,53,54,55,56,60,62,65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_acc_trans_in = pd.read_csv("../data/source/automatic/eutl_transactions.csv")


In [8]:
def assign_account_id(row) -> str:
    if pd.notnull(row["ACCOUNT_IDENTIFIER"]):
        if pd.notnull(row["registry_id"]):
            registry_id = row["registry_id"]
        else:
            registry_id = "UNKNOWN"
        return registry_id + "_" + str(int(row["ACCOUNT_IDENTIFIER"]))


df_acc_trans = df_acc_trans_in.copy()
# extract account involved in transactions
lst_df = []
for prefix in ["TRANSFERRING", "ACQUIRING"]:
    cols = [c for c in df_acc_trans.columns if c.startswith(prefix)]
    df_ = df_acc_trans[cols].copy()
    cols = [c.replace(f"{prefix}_", "") for c in cols]
    df_.columns = cols
    lst_df.append(df_)
df_acc_trans = (
    pd.concat(lst_df, axis=0)
    .drop_duplicates()
    .reset_index(drop=True)
    .assign(
        registry_id=lambda df: df["REGISTRY_NAME"].str.strip().map(map_registry_names),
        account_id=lambda df: df.apply(assign_account_id, axis=1),
    )
)
# df_acc_trans.info()

## How do direct downloads compare against power bi accounts?

Overall we observe:

1. Only direct downloads provide information on dates
2. Only the PowerBi downloads provide information on the account holder 

In [9]:
acc_only_direct = set(df_acc_direct["account_id"]) - set(df_acc_bi["account_id"])
acc_only_bi = set(df_acc_bi["account_id"]) - set(df_acc_direct["account_id"])
acc_only_in_trans = set(df_acc_trans["account_id"]) - set(df_acc_direct["account_id"])
print(f"Accounts only in direct download: {len(acc_only_direct)}")
print(f"Accounts only in BI download: {len(acc_only_bi)}")
print(f"Accounts only in transactions download: {len(acc_only_in_trans)}")

Accounts only in direct download: 277
Accounts only in BI download: 0
Accounts only in transactions download: 744


### Account types

#### Power BI

Account types are aggregated by groups and more disaggregated in the Power BI app. 
However, the information is somewhat mixed across columns. The pattern seems to be:

1. Holding accounts: 
   1. ACCOUNT_TYPE: is Holding Account
   2. ETS_ACCOUNT_TYPE: provides the detailed account
   3. FULL_TYPE: Repeats ETS_ACCOUNT_TYPE (Not for all)
2. Other account:
   1. ETS_ACCOUNT_TYPE: missing
   2. FULL_TYPE: Is more detailed

In [10]:
df_ = df_acc_direct[["ACCOUNT_TYPE", "ETS_ACCOUNT_TYPE", "FULL_TYPE"]].drop_duplicates()
print("Differences in ACCOUNT_TYPE vs FULL_TYPE:")
df_[
    (df_["ACCOUNT_TYPE"].str.strip() != df_["FULL_TYPE"].str.strip())
    & (df_["ACCOUNT_TYPE"] != "Holding Account")
].drop_duplicates()

Differences in ACCOUNT_TYPE vs FULL_TYPE:


,ACCOUNT_TYPE,ETS_ACCOUNT_TYPE,FULL_TYPE
0,Operator Holding Account,NaN,Former Operator Holding Account
15671,Non-Kyoto Account Type,Verifier Account,Verifier Account


In [11]:
# df_acc_bi[["Account Type"]].drop_duplicates()

#### Differences Power Bi and direct


We need to check the differences in the account types:

As we can see, the account types are complete in the directly donwloaded table.
We can use the ETS_ACCOUNT_TYPE as basis and fill ot from FULL_TYPE.

In [12]:
df_acc = df_acc_direct.merge(df_acc_bi, on="account_id", how="outer")
direct, bi = "FULL_TYPE", "Account Type"
df_ = df_acc[(df_acc[direct].str.strip() != df_acc[bi].str.strip())]
print(f"{len(df_)} differing account types between direct downloads and Power BI")
df_ = (
    df_[["ACCOUNT_TYPE", "ETS_ACCOUNT_TYPE"] + [direct, bi]]
    .sort_values(by=direct)
    .drop_duplicates()
)
df_

281 differing account types between direct downloads and Power BI


,ACCOUNT_TYPE,ETS_ACCOUNT_TYPE,FULL_TYPE,Account Type
14024,Holding Account,AEA Deletion Account,AEA Deletion Account,NaN
14023,Holding Account,AEA Total quantity Account,AEA Total quantity Account,NaN
14128,Holding Account,ESD Compliance Account,ESD Compliance Account,NaN
2488,Operator Holding Account,NaN,Former Operator Holding Account,NaN
36729,Holding Account,NaN,Party Holding Account,NaN
2487,Person Account in National Registry,NaN,Person Account in National Registry,NaN
2503,Retirement Account,NaN,Retirement Account,NaN
2502,Voluntary Cancellation Account (Type 3),NaN,Voluntary Cancellation Account (Type 3),NaN
680,Holding Account,NaN,NaN,NaN
18378,Holding Account,International Credit Account,NaN,International Credit Account


In [13]:
df_acc.iloc[18378:18381]

,ACCOUNT_IDENTIFIER,REGISTRY_CODE,REGISTRY_NAME,ACCOUNT_NAME,ACCOUNT_TYPE,ETS_ACCOUNT_TYPE,FULL_TYPE,OPEN_DATE,END_OF_VALIDITY_DATE,IS_CLOSURE_PENDING,...,Account Name,Installation/Aircraft Operator/Maritime Operator ID,Company Registration No,Main Address Line,City,Legal Entity Identifier,Telephone 1,Telephone 2,Email,registry_id
18378,5022590,EU,European Commission,EU International Credit Account,Holding Account,International Credit Account,NaN,2014-01-29,9999-12-31,N,...,EU International Credit Account,NaN,NaN,Av d'Auderghem 19,Brussels,NaN,NaN,NaN,NaN,EU
18379,5022591,EU,European Commission,EU International Credit Account - Aviation,Holding Account,International Credit Account,NaN,2014-01-29,9999-12-31,N,...,EU International Credit Account - Aviation,NaN,NaN,Av d'Auderghem 19,Brussels,NaN,NaN,NaN,NaN,EU
18380,5022592,EU,European Commission,EU Credit Exchange Account - Aviation,Holding Account,Credit Exchange Account,NaN,2014-01-29,9999-12-31,N,...,EU Credit Exchange Account - Aviation,NaN,NaN,Av d'Auderghem 19,Brussels,NaN,NaN,NaN,NaN,EU


#### Account Names

Comparing names between direct and power BI downloads, it seems that the BI downloads
(the column Account Name) have been normalized. It also appears that the BI downloads 
have more missing values. We therefore prefer the direct downloads.

In [14]:
df_acc = df_acc_direct.merge(df_acc_bi, on="account_id", how="outer")
to_check = {
    "Account Name": "ACCOUNT_NAME",
}

for right, left in to_check.items():
    df_ = df_acc[(df_acc[left].str.strip() != df_acc[right].str.strip())]
    if not df_.empty:
        df_ = df_[[left, right]].sort_values(by=left).drop_duplicates()
        break
df_acc[[left, right]].drop_duplicates().sort_values(by=left)
df_.info()
df_

<class 'pandas.core.frame.DataFrame'>
Index: 542 entries, 13293 to 47607
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ACCOUNT_NAME  542 non-null    object
 1   Account Name  265 non-null    object
dtypes: object(2)
memory usage: 12.7+ KB


,ACCOUNT_NAME,Account Name
13293,A/S GLOBAL RISK MANAGEMENT LTD. HOLDING,A/S Global Risk Management Ltd. Holding
28081,A2A S.p.A.,A2A S.P.A.
34774,A2A Trading,a2a Trading
22613,ABN AMRO BANK N.v.,ABN AMRO BANK N.V.
39253,ABN AMRO Bank N.V.,ABN AMRO BANK N.V.
...,...,...
33235,stabilimento di Brindisi,Stabilimento di Brindisi
11877,ubr logistik,UBR Logistik
34790,veronagest,Veronagest
5641,vertus energiehandel gmbh,Vertus Energiehandel Gmbh


## How do transaction data compare against direct downloads

There are plenty of accounts that are only in the direct downloads but not in the transaction data. This is natural
as there are accounts that do not transfer allowances. There are also accounts, that are in the transaction data
but not in the downloaded account table. 

Closer inspection reveals that these are mainly accounts that are not registered 
in the European system but involved in transactions. These are accounts are pretty
raw in the sense that they do not provide account names or any information.

In [15]:
acc_only_direct = set(df_acc_direct["account_id"]) - set(df_acc_trans["account_id"])
acc_only_in_trans = set(df_acc_trans["account_id"]) - set(df_acc_direct["account_id"])
print(f"Accounts only in direct download: {len(acc_only_direct)}")
print(f"Accounts only in transactions download: {len(acc_only_in_trans)}")

Accounts only in direct download: 13355
Accounts only in transactions download: 744


In [16]:
df_ = df_acc_trans[df_acc_trans["account_id"].isin(acc_only_in_trans)].sort_values(
    by="account_id"
)
df_.info()
df_

<class 'pandas.core.frame.DataFrame'>
Index: 790 entries, 13493 to 34370
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   REGISTRY_NAME                               790 non-null    object 
 1   ACCOUNT_TYPE1                               760 non-null    float64
 2   ACCOUNT_TYPE2                               790 non-null    object 
 3   ACCOUNT_TYPE3                               790 non-null    object 
 4   ACCOUNT_OPEN_DT                             0 non-null      object 
 5   ACCOUNT_END_OF_VALIDITY                     0 non-null      object 
 6   ACCOUNT_NAME                                0 non-null      object 
 7   ACCOUNT_IDENTIFIER                          760 non-null    float64
 8   ACCOUNT_HOLDER                              0 non-null      object 
 9   ACCOUNT_HOLDER_ADDRESS1                     0 non-null      object 
 10  ACCOUNT_HOLDE

,REGISTRY_NAME,ACCOUNT_TYPE1,ACCOUNT_TYPE2,ACCOUNT_TYPE3,ACCOUNT_OPEN_DT,ACCOUNT_END_OF_VALIDITY,ACCOUNT_NAME,ACCOUNT_IDENTIFIER,ACCOUNT_HOLDER,ACCOUNT_HOLDER_ADDRESS1,...,INSTALLATION_PARENT_COMPANY,INSTALLATION_SUBSIDIARY_COMPANY,INSTALLATION_EPER_IDENTIFICATION,INSTALLATION_CITY,INSTALLATION_POSTAL_CODE,INSTALLATION_ADDRESS1,INSTALLATION_ADDRESS2,INSTALLATION_MAIN_ACTIVITY,registry_id,account_id
13493,CDM,110.0,-,-,NaN,NaN,NaN,1000.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_1000
22157,CDM,100.0,-,-,NaN,NaN,NaN,1004.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_1004
27979,CDM,100.0,-,-,NaN,NaN,NaN,2004.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_2004
22201,CDM,100.0,-,-,NaN,NaN,NaN,2005.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_2005
22289,CDM,100.0,-,-,NaN,NaN,NaN,2006.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_2006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26923,European Commission,NaN,-,-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,EU,None
27354,Norway,NaN,-,-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NO,None
30322,Estonia,NaN,-,-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,EE,None
31437,Liechtenstein,NaN,-,-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,LI,None


## Unify accounts

To get a single account table we follow the following strategy:

1. The direct download is our base table
2. Account Types:
   1. SUPPLEMENTARY_ACCOUNT_TYPE:
      1. ETS_ACCOUNT_TYPE is the base column
      2. Missings are filled from "FULL_TYPE"
      3. Remaining missings are filled from power bi table column "Account Type"
3. Account Name:
   1. direct download ACCOUNT_NAME is the base column
4. Installation Mapping: 
   1. The mapping is already given in the installation data. So for the first moment we ignore.
   2. Later we need to check for consistency and whether there is additional information
5. Account Holder Mapping
   1. Account holders are not provided as separate table
   2. Extracted from BI data and enhanced by 


In [17]:
from pathlib import Path


def load_account_data(
    fn_direct: str | Path,
    fn_bi: str | Path,
    fn_trans: str | Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load account data from multiple sources into DataFrames.

    Args:
        fn_direct (str | Path): File path for direct download CSV.
        fn_bi (str | Path): File path for BI Excel
        fn_trans (str | Path): File path for transaction data CSV

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
            DataFrames for direct, BI, and transaction account data.
    """
    # get automatically downloaded account data
    df_direct = pd.read_csv(fn_direct).assign(
        account_id=lambda df: df["REGISTRY_CODE"]
        + "_"
        + df["ACCOUNT_IDENTIFIER"].astype(str),
    )

    # create mapping of registry names to codes
    map_registry_names = df_direct.set_index("REGISTRY_NAME")["REGISTRY_CODE"].to_dict()
    map_registry_names.update({"Switzerland": "CH", "CDM": "CDM"})

    # extract account data from BI download
    df_bi = (
        pd.read_excel(
            fn_bi,
            skipfooter=2,
            engine="calamine",
            na_values=["-"],
            keep_default_na=True,
        )
        .rename(columns={"..1": "registry_id"})
        .drop(columns=".")
        .assign(
            account_id=lambda df: df["registry_id"]
            + "_"
            + df["Account Identifier"].astype(str),
        )
    )

    # extract account data from transactions
    def assign_account_id(row) -> str:
        if pd.notnull(row["ACCOUNT_IDENTIFIER"]):
            if pd.notnull(row["registry_id"]):
                registry_id = row["registry_id"]
            else:
                registry_id = "UNKNOWN"
            return registry_id + "_" + str(int(row["ACCOUNT_IDENTIFIER"]))

    df_trans = pd.read_csv(fn_trans, low_memory=False)
    lst_df = []
    for prefix in ["TRANSFERRING", "ACQUIRING"]:
        cols = [c for c in df_trans.columns if c.startswith(prefix)]
        df_ = df_trans[cols].copy()
        cols = [c.replace(f"{prefix}_", "") for c in cols]
        df_.columns = cols
        lst_df.append(df_)
    df_trans = (
        pd.concat(lst_df, axis=0)
        .drop_duplicates()
        .reset_index(drop=True)
        .assign(
            registry_id=lambda df: df["REGISTRY_NAME"]
            .str.strip()
            .map(map_registry_names),
            account_id=lambda df: df.apply(assign_account_id, axis=1),
        )
    )

    return df_direct, df_bi, df_trans


df_direct, df_bi, df_trans = load_account_data(
    fn_direct="../data/source/automatic/eutl_accounts.csv",
    fn_bi="../data/source/manual/accounts.xlsx",
    fn_trans="../data/source/automatic/eutl_transactions.csv",
)

In [18]:
from pathlib import Path


def unify_accounts(
    df_direct: pd.DataFrame,
    df_bi: pd.DataFrame,
    df_trans: pd.DataFrame,
    fn_out: str | Path | None = None,
) -> pd.DataFrame:
    """Unify account data from different sources into a single DataFrame.

    Args:
        df_direct (pd.DataFrame): DataFrame containing account data from direct download.
        df_bi (pd.DataFrame): DataFrame containing account data from Power BI.
        df_trans (pd.DataFrame): DataFrame containing account data from transactions.
        fn_out (str | Path | None): Optional file path to save the unified DataFrame as CSV.

    Returns:
        pd.DataFrame: Unified DataFrame containing account data from all sources.
    """
    # 1. create the basic table from direct download
    cols = {
        "account_id": "account_id",
        "registry_code": "registry_id",
        "ACCOUNT_NAME": "accountName",
        "OPEN_DATE": "openingDate",
        "END_OF_VALIDITY_DATE": "closingDate",
        "IS_CLOSURE_PENDING": "isClosurePending",
        "SNAPSHOT_DATE": "snapshotDate",
    }
    df_accounts = df_direct.rename(columns=cols).drop(
        columns=["REGISTRY_NAME", "ACCOUNT_IDENTIFIER", "REGISTRY_CODE"]
    )

    # 2. Make a consistent account type column
    df_accounts = df_accounts.assign(
        account_type=(lambda df: df["ETS_ACCOUNT_TYPE"].combine_first(df["FULL_TYPE"]))
    ).drop(columns=["ETS_ACCOUNT_TYPE", "FULL_TYPE", "ACCOUNT_TYPE"])
    return df_accounts


df_accounts = unify_accounts(df_direct, df_bi, df_trans)

In [66]:
df_trans.columns

Index(['REGISTRY_NAME', 'ACCOUNT_TYPE1', 'ACCOUNT_TYPE2', 'ACCOUNT_TYPE3',
       'ACCOUNT_OPEN_DT', 'ACCOUNT_END_OF_VALIDITY', 'ACCOUNT_NAME',
       'ACCOUNT_IDENTIFIER', 'ACCOUNT_HOLDER', 'ACCOUNT_HOLDER_ADDRESS1',
       'ACCOUNT_HOLDER_ADDRESS2', 'ACCOUNT_HOLDER_CITY',
       'ACCOUNT_HOLDER_POSTAL_CODE', 'ACCOUNT_HOLDER_COUNTRY_CODE',
       'ACCOUNT_HOLDER_COMPANY_REGISTRATION_NUMBER', 'ACCOUNT_HOLDER_LEI',
       'INSTALLATION_NAME', 'INSTALLATION_INSTALLATION_IDENTIFIER',
       'INSTALLATION_PERMIT_IDENTIFIER', 'INSTALLATION_PARENT_COMPANY',
       'INSTALLATION_SUBSIDIARY_COMPANY', 'INSTALLATION_EPER_IDENTIFICATION',
       'INSTALLATION_CITY', 'INSTALLATION_POSTAL_CODE',
       'INSTALLATION_ADDRESS1', 'INSTALLATION_ADDRESS2',
       'INSTALLATION_MAIN_ACTIVITY', 'registry_id', 'account_id'],
      dtype='object')

In [78]:
import hashlib
import re


def generate_account_holder_id(row: pd.Series, digits: int = 10) -> str:
    """Generate a unique account holder ID based on available information.

    Args:
        row (pd.Series): A row from the dataframe the function is applied to
        digits (int): Number of digits to use for the hash ID (default: 10)

    Returns:
        str: A stable hash-based unique identifier for the account holder.
    """
    # 1. normalized the account holder name
    name = str(row["accountHolderName"]).strip().lower()

    # 2. Check of company registration number exists
    raw_crn = str(row["companyRegistrationNumber"]).strip().lower()

    is_missing = (
        pd.isnull(raw_crn)
        or raw_crn in ["", "nan", "none", "null"]
        or re.match(r"^0+$", raw_crn)
        or re.match(r"^-+$", raw_crn)
    )

    crn = "no_crn" if is_missing else raw_crn

    # 3. Create composite ID string
    composite_id = f"{name}|{crn}"

    # 4. Generate stable hash-based ID
    hash = hashlib.sha256(composite_id.encode()).hexdigest()

    return hash[:digits]

In [101]:
def extract_holders(
    df_trans: pd.DataFrame, df_bi: pd.DataFrame, digits: int = 10
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Extract unique account holders from transaction data. Holders are only extracted
        if they account holder name is available. We extract first from the
        transaction data, as they have more complete holder information. Afterwards,
        we fill in missing holders from the BI data. The holder IDs are generated
        using a stable hash-based method based on account holder name and company
        registration number.

    Args:
        df_trans: DataFrame containing accounts from the transaction data.
        df_bi: DataFrame containing accounts from the BI data.
        digits: Number of digits to use for the hash ID (default: 10)

    Returns:

        A tuple containing two DataFrames:
            - DataFrame of unique account holders with generated holder IDs.
            - DataFrame of accounts IDs with associated holder IDs.
    """
    # 1. Extract holders from transaction data
    trans_holder_cols = {
        "ACCOUNT_HOLDER": "accountHolderName",
        "ACCOUNT_HOLDER_COMPANY_REGISTRATION_NUMBER": "companyRegistrationNumber",
        "ACCOUNT_HOLDER_LEI": "legalEntityIdentifier",
        "ACCOUNT_HOLDER_ADDRESS1": "addressMain",
        "ACCOUNT_HOLDER_ADDRESS2": "addressSecondary",
        "ACCOUNT_HOLDER_POSTAL_CODE": "postalCode",
        "ACCOUNT_HOLDER_CITY": "city",
        "ACCOUNT_HOLDER_COUNTRY_CODE": "country",
    }
    df_holder_trans = (
        df_trans[["account_id"] + list(trans_holder_cols.keys())]
        # drop holder if the name is missing
        .loc[lambda df: pd.notnull(df["ACCOUNT_HOLDER"])]
        .rename(columns=trans_holder_cols)
        # assign unique id
        .assign(
            holder_id=lambda df: df.apply(generate_account_holder_id, axis=1),
        )
    )
    # determine the accounts linked to each holder
    df_holder_trans = df_holder_trans.drop_duplicates(subset=["holder_id"])

    # 2. Extract holders from BI data
    bi_holder_cols = {
        "Account Holder Name": "accountHolderName",
        "Company Registration No": "companyRegistrationNumber",
        "Legal Entity Identifier": "legalEntityIdentifier",
        "Main Address Line": "addressMain",
        "City": "city",
        "Telephone 1": "telephone1",
        "Telephone 2": "telephone2",
        "Email": "email",
    }
    df_holder_bi = (
        df_bi[["account_id"] + list(bi_holder_cols.keys())]
        # drop holder if the name is missing
        .loc[lambda df: pd.notnull(df["Account Holder Name"])]
        # exclude accounts that are already in the transaction holders
        .loc[lambda df: ~df["account_id"].isin(df_holder_trans["account_id"])]
        .rename(columns=bi_holder_cols)
        # assign unique id
        .assign(
            holder_id=lambda df: df.apply(generate_account_holder_id, axis=1),
        )
    )

    # 3. combine both holder dataframes, drop duplicates and create account-holder
    # mapping
    df_holder = pd.concat([df_holder_bi, df_holder_trans], axis=0)
    df_link_account_holder = df_holder[["holder_id", "account_id"]].drop_duplicates()
    df_holder = df_holder.drop_duplicates(subset=["holder_id"])

    return df_holder.reset_index(drop=True), df_link_account_holder.reset_index(
        drop=True
    )


df_holders, df_link_account_holder = extract_holders(df_trans, df_bi)
df_holders.info()
df_link_account_holder.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22613 entries, 0 to 22612
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   account_id                 22613 non-null  object
 1   accountHolderName          22613 non-null  object
 2   companyRegistrationNumber  19763 non-null  object
 3   legalEntityIdentifier      3272 non-null   object
 4   addressMain                22594 non-null  object
 5   city                       22595 non-null  object
 6   telephone1                 294 non-null    object
 7   telephone2                 205 non-null    object
 8   email                      313 non-null    object
 9   holder_id                  22613 non-null  object
 10  addressSecondary           1666 non-null   object
 11  postalCode                 5562 non-null   object
 12  country                    5567 non-null   object
dtypes: object(13)
memory usage: 2.2+ MB
<class 'pandas.core.frame